# 🚗 Car Crash Severity Prediction
**NETW 1013: Machine Learning — GUC Project**

**Kaggle Competition:** https://www.kaggle.com/competitions/car-crash-severity-prediction

---

## Project Goal
Predict the **severity** of a car crash (e.g., Minor Injury, Severe Injury, Fatal) using structured crash-related features such as speed, weather, vehicle type, driver characteristics, and road conditions.

## Workflow Overview
1. Exploratory Data Analysis (EDA)
2. Pre-processing & Feature Engineering
3. Model Training (KNN, Decision Tree, Random Forest, Logistic Regression, Naïve Bayes + extras)
4. Evaluation & Comparison
5. Final Submission to Kaggle

---
## 0. Imports & Setup

In [1]:
# ── Standard Libraries ──────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Scikit-learn: Pre-processing ─────────────────────────────────────────────
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline

# ── Scikit-learn: Models ─────────────────────────────────────────────────────
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB

# ── Scikit-learn: Metrics ─────────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, roc_auc_score, ConfusionMatrixDisplay
)

# ── XGBoost (extra model) ────────────────────────────────────────────────────
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print("XGBoost not installed — skipping. Run: pip install xgboost")

# ── Plot Style ────────────────────────────────────────────────────────────────
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 14
RANDOM_STATE = 42
print("✅ All libraries loaded successfully.")

XGBoost not installed — skipping. Run: pip install xgboost
✅ All libraries loaded successfully.


---
## 1. Load Dataset

In [ ]:
train_path = r"C:\Users\ganna\Downloads\car_crash_train.csv"
train_df = pd.read_csv(train_path)   # labeled training data

test_path = r"C:\Users\ganna\Downloads\car_crash_test.csv"
test_df  = pd.read_csv(test_path)    # unlabeled test data (for Kaggle submission)

print(f"Training set shape : {train_df.shape}")
print(f"Test set shape     : {test_df.shape}")
train_df.head()

---
## 2. Exploratory Data Analysis (EDA)

### 2.1 Basic Info & Data Types

In [ ]:
print("=== Dataset Info ===")
train_df.info()

In [ ]:
print("=== Statistical Summary ===")
train_df.describe(include='all')

### 2.2 Target Variable Distribution

In [ ]:
target_col = 'Severity'
severity_counts = train_df[target_col].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
severity_counts.plot(kind='bar', ax=axes[0], color=sns.color_palette('Set2'), edgecolor='black')
axes[0].set_title('Crash Severity — Class Distribution')
axes[0].set_xlabel('Severity')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)

# Pie chart
axes[1].pie(severity_counts, labels=severity_counts.index, autopct='%1.1f%%',
            colors=sns.color_palette('Set2'), startangle=90)
axes[1].set_title('Severity Proportion')

plt.tight_layout()
plt.show()

print("\nClass counts:")
print(severity_counts)

# Comment:
# Observing whether the dataset is balanced or imbalanced is critical.
# If one class (e.g., 'Fatal') is heavily under-represented, we may need
# to apply class_weight='balanced' in our models or use oversampling.

### 2.3 Missing Values

In [ ]:
missing = train_df.isnull().sum()
missing_pct = (missing / len(train_df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

if missing_df.empty:
    print("✅ No missing values found in the training set.")
else:
    print(missing_df)
    
    # Visualize
    missing_df['Missing %'].plot(kind='bar', color='tomato', edgecolor='black')
    plt.title('Missing Values by Feature (%)')
    plt.ylabel('Missing %')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

### 2.4 Numerical Feature Distributions

In [ ]:
numerical_cols = train_df.select_dtypes(include=['int64', 'float64']).columns.tolist()
if target_col in numerical_cols:
    numerical_cols.remove(target_col)

n_cols = 3
n_rows = int(np.ceil(len(numerical_cols) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    axes[i].hist(train_df[col].dropna(), bins=30, color='steelblue', edgecolor='black', alpha=0.8)
    axes[i].set_title(col)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Numerical Feature Distributions', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

# Comment:
# We check for skewness, outliers, and value ranges.
# Highly skewed features may benefit from log-transformation.
# Features on vastly different scales need standardization for distance-based models (KNN, LR).

### 2.5 Categorical Feature Distributions

In [ ]:
categorical_cols = train_df.select_dtypes(include=['object']).columns.tolist()
if target_col in categorical_cols:
    categorical_cols.remove(target_col)

n_cols = 2
n_rows = int(np.ceil(len(categorical_cols) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    counts = train_df[col].value_counts()
    axes[i].bar(counts.index, counts.values,
                color=sns.color_palette('pastel', len(counts)), edgecolor='black')
    axes[i].set_title(col)
    axes[i].set_xlabel('Category')
    axes[i].set_ylabel('Count')
    axes[i].tick_params(axis='x', rotation=30)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Categorical Feature Distributions', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

### 2.6 Crash Speed vs. Severity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
train_df.boxplot(column='Crash Speed (km/h)', by=target_col, ax=axes[0],
                 patch_artist=True, notch=True)
axes[0].set_title('Crash Speed by Severity')
axes[0].set_xlabel('Severity')
axes[0].set_ylabel('Crash Speed (km/h)')
plt.sca(axes[0])
plt.xticks(rotation=30)

# Violin plot
sns.violinplot(x=target_col, y='Crash Speed (km/h)', data=train_df,
               ax=axes[1], palette='Set2')
axes[1].set_title('Crash Speed Distribution by Severity (Violin)')
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle('', fontsize=1)
plt.tight_layout()
plt.show()

# Comment:
# Higher crash speeds intuitively correlate with more severe outcomes.
# If this pattern is confirmed, Crash Speed will likely be a top feature.

### 2.7 Correlation Heatmap (Numerical Features)

In [ ]:
plt.figure(figsize=(12, 8))
corr_matrix = train_df[numerical_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, linewidths=0.5,
            annot_kws={'size': 9})
plt.title('Correlation Heatmap — Numerical Features')
plt.tight_layout()
plt.show()

# Comment:
# Highly correlated features (|r| > 0.85) can cause multicollinearity in
# Logistic Regression and inflate model complexity. We check for such pairs.

### 2.8 Feature vs. Severity — Categorical Cross-tabs

In [ ]:
# Pick a few important categorical features to explore against severity
important_cats = [c for c in ['Weather Condition', 'Road Condition', 'Crash Type',
                               'Vehicle Type', 'Distraction Level', 'Time of Day']
                  if c in categorical_cols]

for col in important_cats:
    ct = pd.crosstab(train_df[col], train_df[target_col], normalize='index') * 100
    ct.plot(kind='bar', stacked=True, figsize=(10, 4),
            colormap='Set2', edgecolor='black')
    plt.title(f'{col} vs. Severity (% within each category)')
    plt.ylabel('Percentage')
    plt.xlabel(col)
    plt.legend(title='Severity', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

### 2.9 EDA Summary

> **Key findings from EDA:**
> - **(Add your observations here after running the cells above)**
> - Example: "Higher crash speeds tend to correlate with Fatal and Severe Injury outcomes."
> - Example: "Icy road conditions show a higher proportion of severe crashes."
> - Example: "The dataset is moderately imbalanced — 'Fatal' is the minority class."
> - These insights will guide our feature engineering and model choices.

---
## 3. Pre-processing

### 3.1 Combine Train + Test for Consistent Encoding

In [ ]:
# Separate target before combining
y_raw = train_df[target_col].copy()

# Drop target from train; mark rows to split later
train_nrows = len(train_df)
combined = pd.concat([train_df.drop(columns=[target_col]), test_df], ignore_index=True)

print(f"Combined shape: {combined.shape}")

### 3.2 Handle Missing Values

In [ ]:
# Numerical: fill with median
num_cols_combined = combined.select_dtypes(include=['int64', 'float64']).columns
for col in num_cols_combined:
    median_val = combined[col].median()
    combined[col].fillna(median_val, inplace=True)

# Categorical: fill with mode
cat_cols_combined = combined.select_dtypes(include=['object']).columns
for col in cat_cols_combined:
    mode_val = combined[col].mode()[0]
    combined[col].fillna(mode_val, inplace=True)

print(f"Remaining missing values: {combined.isnull().sum().sum()}")
# ✅ Expected: 0

### 3.3 Encode Categorical Features

In [ ]:
# We use Label Encoding for binary features (Yes/No, Good/Worn out)
# and One-Hot Encoding for multi-class nominal features.

# ── Binary features (Label Encode) ───────────────────────────────────────────
binary_features = []
for col in cat_cols_combined:
    if combined[col].nunique() == 2:
        binary_features.append(col)
        le = LabelEncoder()
        combined[col] = le.fit_transform(combined[col])
        print(f"Label encoded: {col} → {le.classes_}")

# ── Multi-class nominal features (One-Hot Encode) ────────────────────────────
remaining_cats = [c for c in cat_cols_combined if c not in binary_features]
print(f"\nOne-hot encoding: {remaining_cats}")
combined = pd.get_dummies(combined, columns=remaining_cats, drop_first=True)

print(f"\nDataset shape after encoding: {combined.shape}")

### 3.4 Encode Target Variable

In [ ]:
label_enc_target = LabelEncoder()
y = label_enc_target.fit_transform(y_raw)

print("Target classes:", label_enc_target.classes_)
print("Encoded values:", np.unique(y))

### 3.5 Split Back into Train / Test

In [ ]:
X_all   = combined.iloc[:train_nrows].copy()    # for final model training
X_kaggle = combined.iloc[train_nrows:].copy()   # for Kaggle submission

# ── Train / Validation split ─────────────────────────────────────────────────
X_train, X_val, y_train, y_val = train_test_split(
    X_all, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f"X_train : {X_train.shape}, y_train : {y_train.shape}")
print(f"X_val   : {X_val.shape},   y_val   : {y_val.shape}")
print(f"X_kaggle: {X_kaggle.shape}")

### 3.6 Feature Scaling

In [ ]:
# StandardScaler is needed for KNN and Logistic Regression (distance/gradient-based).
# Tree-based models (DT, RF, NB) are scale-invariant, but we apply scaling globally
# for convenience. We fit ONLY on training data to avoid data leakage.

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_all_sc   = scaler.transform(X_all)
X_kaggle_sc = scaler.transform(X_kaggle)

print("✅ Scaling complete. Fit only on training set to prevent data leakage.")

---
## 4. Model Training & Evaluation

We use a helper function to avoid code repetition and ensure consistent evaluation.

In [ ]:
results = {}   # store metrics for final comparison

def evaluate_model(name, model, X_tr, y_tr, X_v, y_v, use_scaled=True):
    """
    Train the model, predict on validation set, and print a full report.
    Stores accuracy, macro-F1, and val predictions in global `results` dict.
    """
    # Select correct data version
    Xtr = X_train_sc if use_scaled else X_train
    Xv  = X_val_sc   if use_scaled else X_val

    model.fit(Xtr, y_tr)
    y_pred = model.predict(Xv)

    acc  = accuracy_score(y_v, y_pred)
    f1   = f1_score(y_v, y_pred, average='macro', zero_division=0)
    cv   = cross_val_score(model, Xtr, y_tr, cv=5, scoring='accuracy').mean()

    print(f"\n{'='*60}")
    print(f"  Model : {name}")
    print(f"{'='*60}")
    print(f"  Validation Accuracy   : {acc:.4f}")
    print(f"  Macro F1-Score        : {f1:.4f}")
    print(f"  5-Fold CV Accuracy    : {cv:.4f}")
    print()
    print(classification_report(y_v, y_pred,
                                 target_names=label_enc_target.classes_,
                                 zero_division=0))

    # Confusion matrix
    cm = confusion_matrix(y_v, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=label_enc_target.classes_)
    disp.plot(cmap='Blues', xticks_rotation=30)
    plt.title(f'Confusion Matrix — {name}')
    plt.tight_layout()
    plt.show()

    results[name] = {'accuracy': acc, 'macro_f1': f1, 'cv_accuracy': cv,
                     'model': model, 'y_pred': y_pred, 'scaled': use_scaled}
    return model

### 4.1 K-Nearest Neighbors (KNN)

In [ ]:
# ── Hyperparameter search for best K ─────────────────────────────────────────
k_range = range(1, 21)
k_scores = []

for k in k_range:
    knn_temp = KNeighborsClassifier(n_neighbors=k)
    score = cross_val_score(knn_temp, X_train_sc, y_train, cv=5, scoring='accuracy').mean()
    k_scores.append(score)

best_k = k_range[np.argmax(k_scores)]

plt.plot(k_range, k_scores, marker='o', color='steelblue')
plt.axvline(x=best_k, color='red', linestyle='--', label=f'Best K={best_k}')
plt.title('KNN: 5-Fold CV Accuracy vs. K')
plt.xlabel('K')
plt.ylabel('CV Accuracy')
plt.legend()
plt.tight_layout()
plt.show()
print(f"Best K = {best_k} with CV accuracy = {max(k_scores):.4f}")

In [ ]:
# ── Train KNN with best K ─────────────────────────────────────────────────────
knn = KNeighborsClassifier(n_neighbors=best_k)
evaluate_model('KNN', knn, X_train_sc, y_train, X_val_sc, y_val, use_scaled=True)

# Comment:
# KNN is a lazy learner — it stores all training points and classifies
# new points by majority vote among K nearest neighbors.
# It is sensitive to feature scale (hence StandardScaler is essential here)
# and performs poorly with many irrelevant features or noisy data.

### 4.2 Decision Tree

In [ ]:
# ── Tune max_depth ────────────────────────────────────────────────────────────
depth_range = range(2, 20)
dt_scores = []

for d in depth_range:
    dt_temp = DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE)
    score = cross_val_score(dt_temp, X_train, y_train, cv=5, scoring='accuracy').mean()
    dt_scores.append(score)

best_depth = depth_range[np.argmax(dt_scores)]

plt.plot(depth_range, dt_scores, marker='s', color='darkorange')
plt.axvline(x=best_depth, color='red', linestyle='--', label=f'Best depth={best_depth}')
plt.title('Decision Tree: CV Accuracy vs. Max Depth')
plt.xlabel('Max Depth')
plt.ylabel('CV Accuracy')
plt.legend()
plt.tight_layout()
plt.show()
print(f"Best depth = {best_depth}")

In [ ]:
dt = DecisionTreeClassifier(
    max_depth=best_depth,
    criterion='gini',
    min_samples_split=5,
    min_samples_leaf=3,
    random_state=RANDOM_STATE
)
evaluate_model('Decision Tree', dt, X_train, y_train, X_val, y_val, use_scaled=False)

# Comment:
# Decision Trees split data greedily using Gini impurity (or entropy).
# Shallow trees underfit; deep trees overfit. max_depth controls this tradeoff.
# They are interpretable and scale-invariant.

In [ ]:
# ── Visualize the top 3 levels of the Decision Tree ──────────────────────────
plt.figure(figsize=(20, 8))
plot_tree(
    results['Decision Tree']['model'],
    feature_names=X_train.columns.tolist(),
    class_names=label_enc_target.classes_,
    filled=True, max_depth=3, fontsize=9, rounded=True
)
plt.title('Decision Tree (Top 3 Levels)')
plt.tight_layout()
plt.show()

### 4.3 Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,          # grow full trees (bagging handles overfitting)
    min_samples_leaf=2,
    class_weight='balanced', # handles class imbalance
    n_jobs=-1,
    random_state=RANDOM_STATE
)
evaluate_model('Random Forest', rf, X_train, y_train, X_val, y_val, use_scaled=False)

# Comment:
# Random Forest is an ensemble of Decision Trees trained on random subsets
# of data (bagging) and random subsets of features at each split.
# This reduces overfitting and variance significantly compared to a single tree.
# It is typically one of the strongest baseline models for tabular data.

In [ ]:
# ── Feature Importances ───────────────────────────────────────────────────────
rf_model = results['Random Forest']['model']
importances = pd.Series(rf_model.feature_importances_, index=X_train.columns)
top_features = importances.nlargest(15)

top_features.sort_values().plot(kind='barh', figsize=(10, 6), color='seagreen', edgecolor='black')
plt.title('Top 15 Feature Importances — Random Forest')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print("Top 5 most important features:")
print(top_features.head())

# Comment:
# Features with high importance have the strongest predictive power.
# This also helps us understand which factors most influence crash severity.

### 4.4 Logistic Regression

In [ ]:
lr = LogisticRegression(
    multi_class='multinomial',
    solver='lbfgs',
    max_iter=1000,
    C=1.0,               # regularization strength (smaller = stronger)
    class_weight='balanced',
    random_state=RANDOM_STATE
)
evaluate_model('Logistic Regression', lr, X_train_sc, y_train, X_val_sc, y_val, use_scaled=True)

# Comment:
# Logistic Regression models the probability of each class using a
# log-odds linear combination of features. It requires feature scaling.
# 'multinomial' mode extends binary LR to multi-class classification.
# C controls regularization: lower C → stronger L2 penalty → simpler model.

### 4.5 Naïve Bayes

In [ ]:
# GaussianNB assumes features follow a Gaussian (normal) distribution within each class.
# It is fast and works well with continuous numerical features.

nb = GaussianNB()
evaluate_model('Naïve Bayes', nb, X_train_sc, y_train, X_val_sc, y_val, use_scaled=True)

# Comment:
# Naïve Bayes applies Bayes' theorem with the strong (naïve) assumption that
# features are conditionally independent given the class label.
# Despite this simplistic assumption, NB often performs surprisingly well,
# especially with small datasets. It is extremely fast to train and predict.

### 4.6 Neural Network (MLP)

In [ ]:
# ── Multi-layer Perceptron Classifier ────────────────────────────────────────
nn = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    alpha=1e-4,
    batch_size=64,
    learning_rate='adaptive',
    max_iter=300,
    early_stopping=True,
    n_iter_no_change=20,
    random_state=RANDOM_STATE,
    verbose=False
)
evaluate_model('Neural Network (MLP)', nn, X_train_sc, y_train, X_val_sc, y_val, use_scaled=True)

# Comment:
# A neural network learns non-linear feature interactions through multiple
# hidden layers and activation functions.
# StandardScaler is essential here because the MLP uses gradient-based
# optimization and is sensitive to input scale.

---
## 5. Model Comparison

In [ ]:
comparison_df = pd.DataFrame([
    {
        'Model'       : name,
        'Val Accuracy': info['accuracy'],
        'Macro F1'    : info['macro_f1'],
        '5-Fold CV'   : info['cv_accuracy']
    }
    for name, info in results.items()
]).sort_values('Val Accuracy', ascending=False).reset_index(drop=True)

print(comparison_df.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(comparison_df))
width = 0.28

bars1 = ax.bar(x - width, comparison_df['Val Accuracy'], width, label='Val Accuracy', color='steelblue', edgecolor='black')
bars2 = ax.bar(x,         comparison_df['Macro F1'],    width, label='Macro F1',     color='darkorange', edgecolor='black')
bars3 = ax.bar(x + width, comparison_df['5-Fold CV'],   width, label='5-Fold CV',    color='seagreen',   edgecolor='black')

ax.set_xticks(x)
ax.set_xticklabels(comparison_df['Model'], rotation=25, ha='right')
ax.set_ylabel('Score')
ax.set_title('Model Comparison — All Metrics')
ax.set_ylim(0, 1.1)
ax.legend()

# Annotate bars
for bar in list(bars1) + list(bars2) + list(bars3):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
best_model_name = comparison_df.iloc[0]['Model']
print(f"🏆 Best model: {best_model_name}")
print(f"   Val Accuracy : {comparison_df.iloc[0]['Val Accuracy']:.4f}")
print(f"   Macro F1     : {comparison_df.iloc[0]['Macro F1']:.4f}")

# Comment:
# We select the best model based on Validation Accuracy AND Macro F1.
# Macro F1 is especially important here because it treats all classes equally,
# which is critical when the dataset is imbalanced (e.g., few 'Fatal' samples).
# A model with high accuracy but low F1 may be ignoring minority classes.

---
## 6. Re-train Best Model on Full Training Data & Generate Kaggle Submission

In [ ]:
# ── Retrieve and re-instantiate the best model ────────────────────────────────
best_info = results[best_model_name]
best_model = best_info['model']
use_scaled = best_info['scaled']

X_final = X_all_sc   if use_scaled else X_all
X_sub   = X_kaggle_sc if use_scaled else X_kaggle

# Re-train on FULL training set (train + validation)
best_model.fit(X_final, y)
print(f"✅ {best_model_name} re-trained on full {len(X_final)} training samples.")

In [ ]:
# ── Predict on Kaggle test set ────────────────────────────────────────────────
y_kaggle_pred = best_model.predict(X_sub)
y_kaggle_labels = label_enc_target.inverse_transform(y_kaggle_pred)

# ── Build submission file ─────────────────────────────────────────────────────
# Adjust 'id' to whatever the actual ID column is in the test set
id_col = 'id' if 'id' in test_df.columns else test_df.columns[0]

submission = pd.DataFrame({
    id_col   : test_df[id_col],
    'Severity': y_kaggle_labels
})

submission.to_csv('submission.csv', index=False)
print("📄 submission.csv saved!")
print(submission.head(10))
print(f"\nSubmission shape: {submission.shape}")
print("\nPrediction distribution:")
print(submission['Severity'].value_counts())

---
## 7. Final Summary & Conclusions

### 7.1 EDA Takeaways
> *(Fill in after running all cells)*
> - Class distribution: ...
> - Most impactful features: ...
> - Notable patterns found: ...

### 7.2 Pre-processing Decisions
> - Missing values: filled numerical with **median**, categorical with **mode**.
> - Binary categoricals: **Label Encoded**.
> - Multi-class categoricals: **One-Hot Encoded** (drop_first=True to avoid multicollinearity).
> - Scaling: **StandardScaler** applied for KNN and Logistic Regression; tree models used raw encoded features.
> - Target: **Label Encoded** for compatibility with all classifiers.

### 7.3 Model Results Summary

| Model | Val Accuracy | Macro F1 | Notes |
|---|---|---|---|
| KNN | — | — | Sensitive to scale & k |
| Decision Tree | — | — | Interpretable, can overfit |
| Random Forest | — | — | Best baseline ensemble |
| Logistic Regression | — | — | Linear decision boundary |
| Naïve Bayes | — | — | Fast, independence assumption |
| Gradient Boosting | — | — | Sequential ensemble |
| XGBoost | — | — | Optimized boosting |

### 7.4 Best Model
> **[Best Model Name]** achieved the highest Validation Accuracy and Macro F1-Score, making it the most suitable model for predicting car crash severity. It was re-trained on the full training dataset and the predictions were submitted to Kaggle.

### 7.5 Possible Improvements
> - Hyperparameter tuning with `GridSearchCV` or `Optuna` for tree-based models.
> - Oversampling minority class (SMOTE) if class imbalance is severe.
> - Feature selection using RFE or SHAP values.
> - Stacking/blending ensemble of top-performing models.